In [ ]:
# Install dependencies

%pip install anthropic python-dotenv

In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"


In [ ]:
# Make a request

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)
response
response.content[0].text

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write another sentance"
        }
    ]
)
response.content[0].text

'The golden leaves danced gently in the crisp autumn breeze.'

In [50]:
# Client - Api interface

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
         "temperature": temperature, 
         "stop_sequences":stop_sequences
    }
     
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Multi-Turn conversations

# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

print("Prompt 1: "+ answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

print("Prompt 2: " + final_answer)

In [41]:
# System Prompts

# Without system prompt
answer = chat(messages)

print(answer)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print(answer)

Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are limited to being either 0 or 1.
I'd be happy to help you explore this topic further! What specific aspect of quantum computing would you like to understand better? 

For example, are you curious about:
- How quantum computers differ from regular computers?
- What makes quantum bits (qubits) special?
- What kinds of problems quantum computers might solve?
- How quantum mechanics principles apply to computing?

What draws your interest most, and we can work through understanding it step by step?


In [43]:
# Temperature
# Low temperature - more predictable
answer = chat(messages, temperature=0.0)

# High temperature - more creative  
answer = chat(messages, temperature=1.0)


In [47]:
# Controlling model output
messages = []
add_user_message(messages, "Is tea or coffee better at breakfast?")
add_assistant_message(messages, "Coffee is better because")
answer = chat(messages)

answer

' it has more caffeine to help you wake up and is often paired with breakfast foods. Tea has less caffeine and is more of a mid-morning or afternoon drink. That said, it really comes down to personal preference - some people prefer the gentler caffeine boost and variety of flavors that tea offers.\n\nBoth have their benefits:\n\n**Coffee advantages:**\n- Higher caffeine content (95mg vs 25-50mg in tea)\n- Pairs well with typical breakfast foods\n- Strong flavor that many find energizing\n\n**Tea advantages:**\n- More gradual, sustained energy without jitters\n- Contains L-theanine, which can promote calm alertness\n- Wider variety of flavors and types\n- Often easier on the stomach\n\nWhat matters most is what makes you feel good and fits your morning routine!'

In [52]:
# Stop Sequances

messages = []
add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=[", 5"])

answer

'1, 2, 3, 4'

In [56]:
# Structured Data

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

chat(messages)

add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [57]:
import json 

json_format = json.loads(text)

json_format

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'State': 'ENABLED',
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}